In [ ]:
import pandas as pd
from pathlib import Path

YEAR = 2026
data_dir = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path("data")
data_dir.mkdir(exist_ok=True)

In [ ]:
games = pd.read_csv(data_dir / f"games_{YEAR}.csv", index_col=False)
games.head().T

In [ ]:
team_games = pd.read_csv(data_dir / f"team_games_{YEAR}.csv", index_col=False)
team_games.head().T

## Conference Win Percentage Chart

In [ ]:
fbs_conferences = set(team_games.loc[team_games["classification"] == "fbs", "conference"])

nonconf = (
    team_games[team_games["classification"] == "fbs"]
    # lump every FCS opponent into one group (opponent conference isn't an FBS conference)
    .assign(opp_conference=lambda d: d["opp_conference"].where(d["opp_conference"].isin(fbs_conferences), "FCS"))
    .query("conference != opp_conference")
)


def head_to_head(df):
    """Win/loss record of each conference against each opponent conference (drops same-group games)."""
    return (
        df.query("conference != opp_conference")
        .groupby(["conference", "opp_conference"])
        .agg(games=("win", "size"), wins=("win", "sum"))
        .assign(
            losses=lambda d: d["games"] - d["wins"],
            win_pct=lambda d: d["wins"] / d["games"],
        )
        .reset_index()
    )


h2h = head_to_head(nonconf)
h2h.head(15)

In [ ]:
# Matrix view: row conference's win % against column conference (small samples — check games_matrix)
win_pct_matrix = h2h.pivot(index="conference", columns="opp_conference", values="win_pct")
games_matrix = h2h.pivot(index="conference", columns="opp_conference", values="games").fillna(0).astype(int)
win_pct_matrix.style.format("{:.0%}", na_rep="")

### Power 4 vs. Group of 6

Same view, with the Group of 6 conferences collapsed into one group. Games between two G6 conferences drop out (same group).

In [ ]:
G6 = ["American Athletic", "Conference USA", "Mid-American", "Mountain West", "Pac-12", "Sun Belt"]
to_group = lambda s: s.where(~s.isin(G6), "Group of 6")

h2h_g6 = head_to_head(
    nonconf.assign(conference=lambda d: to_group(d["conference"]), opp_conference=lambda d: to_group(d["opp_conference"]))
)
g6_win_pct_matrix = h2h_g6.pivot(index="conference", columns="opp_conference", values="win_pct")
g6_games_matrix = h2h_g6.pivot(index="conference", columns="opp_conference", values="games").fillna(0).astype(int)
display(g6_games_matrix)
g6_win_pct_matrix.style.format("{:.0%}", na_rep="")

### Dot plot: win % by opponent

Each FBS conference's non-conference win % against each Power 4 conference, the Group of 6 (combined), and FCS. Hover for the W–L record.

In [ ]:
import plotly.graph_objects as go

P4 = ["ACC", "Big 12", "Big Ten", "SEC"]

# One series per opponent group, fixed categorical order. Six hues is past what color alone can
# separate on a dot plot, so each series also gets its own marker symbol.
OPP_GROUPS = {
    "vs. ACC":        ("#2a78d6", "circle"),
    "vs. Big 12":     ("#eb6834", "square"),
    "vs. Big Ten":    ("#1baf7a", "diamond"),
    "vs. SEC":        ("#eda100", "triangle-up"),
    "vs. Group of 6": ("#e87ba4", "x"),
    "vs. FCS":        ("#008300", "cross"),
}


def opp_group(conf):
    if conf in P4:
        return f"vs. {conf}"
    if conf in G6:
        return "vs. Group of 6"
    if conf == "FCS":
        return "vs. FCS"
    return None  # FBS Independents as opponents are left out


def win_pct_by(df, col):
    return (
        df.dropna(subset=[col])
        .groupby(["conference", col])
        .agg(games=("win", "size"), wins=("win", "sum"))
        .assign(losses=lambda d: d["games"] - d["wins"], win_pct=lambda d: d["wins"] / d["games"])
        .reset_index()
    )


opp_pct = win_pct_by(nonconf.assign(opp_group=nonconf["opp_conference"].map(opp_group)), "opp_group")

# Order conferences by combined win % vs. all Power 4 opponents (best at top)
vs_p4 = win_pct_by(nonconf.assign(p4=nonconf["opp_conference"].isin(P4).map({True: "P4"})), "p4")
order = vs_p4.sort_values("win_pct")["conference"].tolist()

fig = go.Figure()
for group, (color, symbol) in OPP_GROUPS.items():
    d = opp_pct[opp_pct["opp_group"] == group]
    fig.add_trace(go.Scatter(
        x=d["win_pct"], y=d["conference"], name=group, mode="markers",
        marker=dict(size=9, color=color, symbol=symbol, line=dict(width=1.5, color="#fcfcfb")),
        customdata=d[["wins", "losses"]],
        hovertemplate="<b>%{y}</b> " + group + "<br>%{x:.0%} (%{customdata[0]}–%{customdata[1]})<extra></extra>",
    ))

fig.add_vline(x=0.5, line=dict(color="#c3c2b7", width=1, dash="dot"))
fig.update_layout(
    title=dict(text="Non-conference win % by opponent", font=dict(size=14)),
    template="simple_white",
    plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
    font=dict(color="#0b0b0b", size=11),
    width=700, height=400, margin=dict(l=10, r=10, t=70, b=30),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0, title=None, font=dict(size=10)),
    hoverlabel=dict(bgcolor="white"),
)
fig.update_xaxes(range=[-0.04, 1.04], tickformat=".0%", showgrid=True, gridcolor="#ecebe8", title=None)
fig.update_yaxes(categoryorder="array", categoryarray=order, showgrid=True, gridcolor="#ecebe8", title=None)
fig.show()

### Logo dot plot: one row per conference

Each row is a conference (Group of 6 collapsed into one row); each logo is an opponent at the row's win % against it. G6 and FCS opponents are labeled circles. Ties overlap — hover for the W–L record.

In [ ]:
import base64

LOGO_DIR = data_dir / "logos"  # ESPN conference logos, 96x96 PNG, named like "big_ten.png"


def logo_uri(conf):
    slug = conf.lower().replace(" ", "_").replace("-", "_")
    return "data:image/png;base64," + base64.b64encode((LOGO_DIR / f"{slug}.png").read_bytes()).decode()


# Collapse the G6 conferences into one row; G6 vs. G6 games drop out
grouped = nonconf.assign(conference=to_group(nonconf["conference"]), opp_group=nonconf["opp_conference"].map(opp_group))
logo_pct = win_pct_by(grouped.query("not (conference == 'Group of 6' and opp_group == 'vs. Group of 6')"), "opp_group")
logo_pct["opponent"] = logo_pct["opp_group"].str.removeprefix("vs. ")

# Rows ordered by combined win % vs. Power 4 (best at top)
rows = win_pct_by(grouped.assign(p4=grouped["opp_conference"].isin(P4).map({True: "P4"})), "p4") \
    .sort_values("win_pct")["conference"].tolist()
has_logo = logo_pct["opponent"].isin(P4)  # G6 and FCS have no logo; drawn as labeled circles

# One trace for hover; visible only for the G6 / FCS points
fig = go.Figure(go.Scatter(
    x=logo_pct["win_pct"], y=logo_pct["conference"], mode="markers+text",
    marker=dict(size=28, color="#52514e", opacity=(~has_logo).astype(float)),
    text=logo_pct["opponent"].where(~has_logo, "").replace("Group of 6", "G6"),
    textfont=dict(color="white", size=9),
    customdata=logo_pct[["opp_group", "wins", "losses"]],
    hovertemplate="<b>%{y}</b> %{customdata[0]}<br>%{x:.0%} (%{customdata[1]}–%{customdata[2]})<extra></extra>",
))
for _, r in logo_pct[has_logo].iterrows():
    fig.add_layout_image(
        source=logo_uri(r["opponent"]), xref="x", yref="y", x=r["win_pct"], y=rows.index(r["conference"]),
        sizex=0.05, sizey=0.8, xanchor="center", yanchor="middle", sizing="contain",
    )

fig.add_vline(x=0.5, line=dict(color="#c3c2b7", width=1, dash="dot"))
fig.update_layout(
    title=dict(text="Non-conference win % by opponent", font=dict(size=14)),
    template="simple_white", showlegend=False,
    plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb", font=dict(color="#0b0b0b", size=11),
    width=700, height=380, margin=dict(l=10, r=10, t=50, b=30),
    hoverlabel=dict(bgcolor="white"),
)
fig.update_xaxes(range=[-0.05, 1.05], tickformat=".0%", showgrid=True, gridcolor="#ecebe8", title=None)
fig.update_yaxes(categoryorder="array", categoryarray=rows, range=[-0.5, len(rows) - 0.5],
                 showgrid=True, gridcolor="#ecebe8", showline=False, ticks="", title=None)
fig.show()

In [ ]:
fmt_record = lambda w, l: w.astype(int).astype(str) + "-" + l.astype(int).astype(str)

# Each FBS team's record entering every game (games in kickoff order)
tg = team_games[team_games["classification"] == "fbs"].merge(games[["game_id", "start_date"]], on="game_id").sort_values("start_date")
wins_before = tg.groupby("team")["win"].cumsum() - tg["win"]
record_then = tg.assign(record_then=fmt_record(wins_before, tg.groupby("team").cumcount() - wins_before)) \
    .set_index(["game_id", "team"])["record_then"]

# Current overall record and rank within conference (by conf win %, then overall win %;
# a simple ordering, not official conference tiebreakers). FCS teams get none: we only have their games vs. FBS.
in_conf = (team_games["conference"] == team_games["opp_conference"]) & (team_games["conference"] != "FBS Independents")
standings = (
    team_games[team_games["classification"] == "fbs"]
    .assign(conf_win=team_games["win"] & in_conf, conf_game=in_conf)
    .groupby(["team", "conference"])
    .agg(wins=("win", "sum"), games=("win", "size"), conf_wins=("conf_win", "sum"), conf_games=("conf_game", "sum"))
    .reset_index()
    .assign(
        record_now=lambda d: fmt_record(d["wins"], d["games"] - d["wins"]),
        conf_pct=lambda d: d["conf_wins"] / d["conf_games"].where(d["conf_games"] > 0),
        pct=lambda d: d["wins"] / d["games"],
    )
)
standings["conf_rank"] = (
    standings.sort_values(["conf_pct", "pct"], ascending=False)
    .groupby("conference").cumcount().add(1)
    .where(standings["conference"] != "FBS Independents")
    .astype("Int64")
)
standings = standings.set_index("team")[["record_now", "conf_rank"]]


def games_behind_dot(conference, opponent):
    """Games behind one dot, e.g. games_behind_dot("SEC", "ACC").

    `conference` is a row label ("SEC", "Group of 6", ...); `opponent` is a P4 conference, "Group of 6", or "FCS".
    record_then = record entering the game; record_now / conf_rank = current. opp_* = same for the opponent.
    `conference` in the result is the team's actual conference (e.g. "Mid-American" rather than "Group of 6").
    """
    mask = (grouped["conference"] == conference) & (grouped["opp_group"] == f"vs. {opponent}")
    d = nonconf[mask]  # same rows as `grouped`, but with real (ungrouped) conferences
    return (
        d
        .assign(
            record_then=record_then.reindex(pd.MultiIndex.from_frame(d[["game_id", "team"]])).values,
            opp_record_then=record_then.reindex(pd.MultiIndex.from_frame(d[["game_id", "opponent"]])).values,
        )
        .join(standings, on="team")
        .join(standings.add_prefix("opp_"), on="opponent")
        .drop(columns=["game_id", "classification"])
        .rename(columns={"points_for": "pts", "points_against": "opp_pts"})
        .sort_values(["week", "team"])
        .reset_index(drop=True)
    )

In [ ]:
import ipywidgets as widgets


def ordinal(n):
    return f"{n}{'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"


def standing(rank, conf):
    """'3rd SEC'; just the conference name when there's no rank (Independents, FCS)."""
    return conf if pd.isna(rank) else f"{ordinal(int(rank))} {conf}"


def then_now(then, now):
    return "—" if pd.isna(then) else f"{then} → {now}"


def pretty_games(d):
    score = lambda team, pts: team + " " + pts.astype(int).astype(str)
    table = pd.DataFrame({
        "Wk": d["week"],
        "Matchup": score(d["team"], d["pts"]) + d["home"].map({True: "  vs  ", False: "  @  "}) + score(d["opponent"], d["opp_pts"]),
        "Win": d["win"],
        "Standing": [standing(r, c) for r, c in zip(d["conf_rank"], d["conference"])],
        "Record (then → now)": [then_now(t, n) for t, n in zip(d["record_then"], d["record_now"])],
        "Opp. standing": [standing(r, c) for r, c in zip(d["opp_conf_rank"], d["opp_conference"])],
        "Opp. record (then → now)": [then_now(t, n) for t, n in zip(d["opp_record_then"], d["opp_record_now"])],
    })
    return (
        table.style
        .hide(axis="index")
        .format({"Win": lambda w: "☑" if w else "☐"})
        .set_properties(**{"text-align": "left", "padding": "2px 10px"})
        .set_properties(subset=["Win"], **{"text-align": "center", "font-size": "1.2em"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left"), ("padding", "2px 10px")]}])
    )


conf_dd = widgets.Dropdown(options=rows[::-1], value="SEC", description="Conference")
opp_dd = widgets.Dropdown(options=[g.removeprefix("vs. ") for g in OPP_GROUPS], value="ACC", description="vs.")
games_html = widgets.HTML()  # the table lives here; each change replaces it


def show_games(_=None):
    d = games_behind_dot(conf_dd.value, opp_dd.value)
    games_html.value = (
        f"<p>No {conf_dd.value} games vs. {opp_dd.value}.</p>" if d.empty else pretty_games(d).to_html()
    )


conf_dd.observe(show_games, names="value")
opp_dd.observe(show_games, names="value")
show_games()
widgets.VBox([widgets.HBox([conf_dd, opp_dd]), games_html])